In [14]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/interim/cleaned_rainfall.csv")

In [3]:
df.head()

,SUBDIVISION,YEAR,JAN,FEB,MAR,APR,MAY,JUN,JUL,AUG,SEP,OCT,NOV,DEC,ANNUAL,JF,MAM,JJAS,OND
0,Andaman & Nicobar Islands,1901,49.2,87.1,29.2,2.3,528.8,517.5,365.1,481.1,332.6,388.5,558.2,33.6,3373.2,136.3,560.3,1696.3,980.3
1,Andaman & Nicobar Islands,1902,0.0,159.8,12.2,0.0,446.1,537.1,228.9,753.7,666.2,197.2,359.0,160.5,3520.7,159.8,458.3,2185.9,716.7
2,Andaman & Nicobar Islands,1903,12.7,144.0,0.0,1.0,235.1,479.9,728.4,326.7,339.0,181.2,284.4,225.0,2957.4,156.7,236.1,1874.0,690.6
3,Andaman & Nicobar Islands,1904,9.4,14.7,0.0,202.4,304.5,495.1,502.0,160.1,820.4,222.2,308.7,40.1,3079.6,24.1,506.9,1977.6,571.0
4,Andaman & Nicobar Islands,1905,1.3,0.0,3.3,26.9,279.5,628.7,368.7,330.5,297.0,260.7,25.4,344.7,2566.7,1.3,309.7,1624.9,630.8


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4188 entries, 0 to 4187
Data columns (total 19 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   SUBDIVISION  4188 non-null   object 
 1   YEAR         4188 non-null   int64  
 2   JAN          4188 non-null   float64
 3   FEB          4188 non-null   float64
 4   MAR          4188 non-null   float64
 5   APR          4188 non-null   float64
 6   MAY          4188 non-null   float64
 7   JUN          4188 non-null   float64
 8   JUL          4188 non-null   float64
 9   AUG          4188 non-null   float64
 10  SEP          4188 non-null   float64
 11  OCT          4188 non-null   float64
 12  NOV          4188 non-null   float64
 13  DEC          4188 non-null   float64
 14  ANNUAL       4188 non-null   float64
 15  JF           4188 non-null   float64
 16  MAM          4188 non-null   float64
 17  JJAS         4188 non-null   float64
 18  OND          4188 non-null   float64
dtypes: flo

In [6]:
#F-1: Monsoon Contribution (%): 
# Monsoon rainfall is the primary source of groundwater recharge in most parts of India so we will calculate Monsoon Percentage 

df["Monsoon_Percentage"] = (df["JJAS"] / df["ANNUAL"]) * 100
df['Monsoon_Percentage']

0       50.287561
1       62.087085
2       63.366471
3       64.216132
4       63.306970
          ...    
4183    74.107831
4184    68.709677
4185    52.401242
4186    69.944637
4187    63.735695
Name: Monsoon_Percentage, Length: 4188, dtype: float64

In [7]:
#F-2: Non-Monsoon Rainfall: 
# This separates rainfall received outside the monsoon season. 

df["Non_Monsoon_Rainfall"] = df["ANNUAL"] - df["JJAS"] 
df['Non_Monsoon_Rainfall']

0       1676.9
1       1334.8
2       1083.4
3       1102.0
4        941.8
         ...  
4183     369.3
4184     436.5
4185     782.0
4186     320.3
4187     630.6
Name: Non_Monsoon_Rainfall, Length: 4188, dtype: float64

In [10]:
#F-3: Rainfall Concentration Index (%): 
# This measures how dependent a region is on the monsoon 

df["Rainfall_Concentration"] = (df["JJAS"] / df["ANNUAL"])*100 
df['Rainfall_Concentration']

0       50.287561
1       62.087085
2       63.366471
3       64.216132
4       63.306970
          ...    
4183    74.107831
4184    68.709677
4185    52.401242
4186    69.944637
4187    63.735695
Name: Rainfall_Concentration, Length: 4188, dtype: float64

In [11]:
#F-4: Pre nd Post Monsoon Comparison:
# we will compare pre-monsoon and post-monsoon rainfall

df["Pre_Post_Monsoon_Diff"] = df["MAM"] - df["OND"]
df['Pre_Post_Monsoon_Diff']

0      -420.0
1      -258.4
2      -454.5
3       -64.1
4      -321.1
        ...  
4183    -46.5
4184   -213.8
4185   -331.5
4186    -82.2
4187   -112.9
Name: Pre_Post_Monsoon_Diff, Length: 4188, dtype: float64

In [15]:
#F-5: Previous Year's Rainfall: 

df = df.sort_values(["SUBDIVISION", "YEAR"])

df["Previous_Annual_Rainfall"] = (
    df.groupby("SUBDIVISION")["ANNUAL"]
      .shift(1)
)
df['Previous_Annual_Rainfall']

0          NaN
1       3373.2
2       3520.7
3       2957.4
4       3079.6
         ...  
1259     389.6
1260     932.8
1261     486.9
1262     582.7
1263     692.8
Name: Previous_Annual_Rainfall, Length: 4188, dtype: float64

Even though rainfall is lower, groundwater may still remain relatively high because of the recharge from past year.

So when predicting groundwater for current year, knowing only the rainfall of the current year might not be enough.

Knowing last year's rainfall provides additional information

In [16]:
#F-6: Rainfall Anomaly:
# rainfall anomaly tells us how different is this year's rainfall from the long-term average rainfall of that subdivision

df = df.sort_values(["SUBDIVISION", "YEAR"])

df["Historical_Mean_Rainfall"] = (
    df.groupby("SUBDIVISION")["ANNUAL"]
      .transform(lambda x: x.expanding().mean().shift(1))
)

df["Rainfall_Anomaly"] = (
    df["ANNUAL"] - df["Historical_Mean_Rainfall"]
)

expanding().mean() computes the running mean up to the current row.

shift(1) ensures the current year's rainfall is not included in its own baseline.

In [17]:
df.isnull().sum()

SUBDIVISION                  0
YEAR                         0
JAN                          0
FEB                          0
MAR                          0
APR                          0
MAY                          0
JUN                          0
JUL                          0
AUG                          0
SEP                          0
OCT                          0
NOV                          0
DEC                          0
ANNUAL                       0
JF                           0
MAM                          0
JJAS                         0
OND                          0
Monsoon_Percentage           0
Non_Monsoon_Rainfall         0
Rainfall_Concentration       0
Pre_Post_Monsoon_Diff        0
Rainfall_Trend              36
Previous_Annual_Rainfall    36
Historical_Mean_Rainfall    36
Rainfall_Anomaly            36
dtype: int64

In [18]:
df.drop(columns=["Historical_Mean_Rainfall"], inplace=True) 

#we have dropped 'Historical_Mean_Rainfall' as its not needed for future it was only an intermediate feature to calculate Rainfall Anomaly

In [19]:
df.isnull().sum()

SUBDIVISION                  0
YEAR                         0
JAN                          0
FEB                          0
MAR                          0
APR                          0
MAY                          0
JUN                          0
JUL                          0
AUG                          0
SEP                          0
OCT                          0
NOV                          0
DEC                          0
ANNUAL                       0
JF                           0
MAM                          0
JJAS                         0
OND                          0
Monsoon_Percentage           0
Non_Monsoon_Rainfall         0
Rainfall_Concentration       0
Pre_Post_Monsoon_Diff        0
Rainfall_Trend              36
Previous_Annual_Rainfall    36
Rainfall_Anomaly            36
dtype: int64

In [15]:
#now lets save the featured dataset 

df.to_csv(
    "../../data/processed/featured_rainfall.csv",
    index=False
)